In [0]:
%python
from datetime import datetime
import re
from collections import defaultdict

# =========================
# PATHS
# =========================

bucket = "retail-etl-dwh-lakehouse"

sftp_path    = f"s3://{bucket}/sftp/"
raw_path     = f"s3://{bucket}/raw/"
archive_path = f"s3://{bucket}/archive/"

# =========================
# CURRENT DATE
# =========================

today = datetime.now().strftime("%d%m%Y")

print(f"Today: {today}")

# =========================
# READ FILES FROM SFTP
# =========================

sftp_files = dbutils.fs.ls(sftp_path)

table_files = defaultdict(list)

print("\n=== Files in sftp/ ===")

for f in sftp_files:

    # Example:
    # customers_src_07052026123045.csv

    match = re.match(r"(.+_src)_(\d{14})\.csv", f.name)

    if match:

        table_name = match.group(1)

        timestamp = datetime.strptime(
            match.group(2),
            "%d%m%Y%H%M%S"
        )

        table_files[table_name].append(
            (timestamp, f.path, f.name)
        )

        print(f"✅ {f.name}")

    else:
        print(f"❌ Skipped: {f.name}")

# =========================
# CLEAR RAW FOLDER
# =========================

print("\n=== Clearing raw/ ===")

try:

    raw_files = dbutils.fs.ls(raw_path)

    for f in raw_files:

        if f.name.endswith(".csv"):

            dbutils.fs.rm(f.path)

            print(f"🗑️ Removed: {f.name}")

except Exception as e:

    print("ℹ️ raw/ already empty")

# =========================
# COPY LATEST FILES
# =========================

print("\n=== Copying latest files to raw/ ===")

for table_name, files in table_files.items():

    latest_file = sorted(
        files,
        key=lambda x: x[0],
        reverse=True
    )[0]

    source_path = latest_file[1]
    file_name   = latest_file[2]

    destination = raw_path + file_name

    dbutils.fs.cp(source_path, destination)

    print(f"✅ Copied: {file_name}")

# =========================
# ARCHIVE OLD RAW FILES
# =========================

print("\n=== Archiving files ===")

archive_today = f"{archive_path}{today}/"

try:

    raw_files = dbutils.fs.ls(raw_path)

    for f in raw_files:

        if f.name.endswith(".csv"):

            archive_dest = archive_today + f.name

            dbutils.fs.cp(f.path, archive_dest)

            print(f"📦 Archived: {f.name}")

except Exception as e:

    print(f"ℹ️ Archive skipped: {e}")

# =========================
# FINAL VALIDATION
# =========================

print("\n=== raw/ contains ===")

for f in dbutils.fs.ls(raw_path):

    if f.name.endswith(".csv"):

        print(f"✅ {f.name}")

print("\n=== archive/ contains ===")

try:

    archive_folders = dbutils.fs.ls(archive_path)

    for folder in archive_folders:

        for f in dbutils.fs.ls(folder.path):

            if f.name.endswith(".csv"):

                print(f"📦 {folder.name}{f.name}")

except:

    print("ℹ️ archive/ empty")